In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.dbutils import *
from datetime import datetime, date

## Reading the external table

In [0]:
if spark.catalog.tableExists("ipl_database.silver.bowler_stats"):
    df = spark.table("ipl_database.bronze.bowling_raw").filter(
        to_date(col("updated_at")) >= date.today()
    )
else:
    df = spark.table("ipl_database.bronze.bowling_raw")
# df.display()

## 00 Extracting bowler stats

In [0]:
df = df.withColumn(
  "stats_array",
  from_json(
    col("stats"),
    ArrayType(StringType())
  )
)

# df.display()

### Extracting details from stats array

In [0]:
df = (
  df
  .withColumn(
    "non_valid_deliveries",
    col("stats_array")[0].cast('int')
  )
  .withColumn(
    "wide_deliveries",
    col("stats_array")[1].cast('int')
  )
  .withColumn(
    "sixes_conceded",
    col("stats_array")[2].cast('int')
  )
  .withColumn(
    "fours_conceded",
    col("stats_array")[3].cast('int')
  )
  .withColumn(
    "dot_deliveries",
    col("stats_array")[4].cast('int')
  )
  .withColumn(
    "match_economy",
    col("stats_array")[5].cast('double')
  )
  .withColumn(
    "runs_conceded_in_match",
    col("stats_array")[6].cast('int')
  )
  .withColumn(
    "maiden_overs_in_match",
    col("stats_array")[7].cast('int')
  )
  .withColumn(
    "overs_bowled_in_match",
    col("stats_array")[8].cast('double')
  )
)

# df.display()

## 01 Adding IPL season

In [0]:
df = df.withColumn(
    "season",
    year(to_date(trim(col("date")),"MMMM dd yyyy"))
)
# df.display()

## trim leading and trailing spaces from the columns

In [0]:
df = (
  df
  .withColumn(
    "id",
    trim(col("id"))
  )
  .withColumn(
    "player_name",
    trim(col("bowler"))
  )
  .withColumn(
    "wickets_in_match",
    col("wicket").cast('int')
  )
)

# df.display()

## Creating final silver table

In [0]:
%skip
df.columns

In [0]:
df_fin = df.select(
    'id',
    'player_name',
    'wickets_in_match',
    'runs_conceded_in_match',
    'non_valid_deliveries',
    'wide_deliveries',
    'sixes_conceded',
    'fours_conceded',
    'dot_deliveries',
    'match_economy',
    'maiden_overs_in_match',
    'overs_bowled_in_match',
    'season',
)

In [0]:
# df_fin.display()

## ++ Match UUID

In [0]:
match_uuid = spark.table("ipl_database.silver.match_details")

df_fin1 = df_fin.alias('a') \
  .join(
    match_uuid.alias('b'),
    on=col('a.id') == col('b.id'),
    how='left'
  ) \
  .drop('id') \
  .select('a.*','b.match_UUID')
# df_fin1.display()

In [0]:
try:
    if spark.catalog.tableExists("ipl_database.silver.bowler_stats"):
        df_fin1.createOrReplaceTempView('df_fin1')
        spark.sql('''
                  merge into ipl_database.silver.bowler_stats bs
                  using df_fin1 df
                  on bs.season = df.season and bs.match_UUID = df.match_UUID and bs.player_name = df.player_name
                --   when matched then update set *
                  when not matched then insert *
                  ''')
        # df_fin1.write.format("delta").mode("append").save(f"abfss://silver@ipldatastorageaccount.dfs.core.windows.net/bowling/")
        spark.sql('''
                refresh table ipl_database.silver.bowler_stats
                ''')
        print("Table updated.")
    else:
        df_fin1.write.format("delta").mode("overwrite").saveAsTable("ipl_database.silver.bowler_stats")
        # spark.sql('''
        #         create table ipl_database.silver.bowler_stats
        #         using delta
        #         location "abfss://silver@ipldatastorageaccount.dfs.core.windows.net/bowling/"
        #         ''')
        print("Table created.")
except Exception as e:
    print(f"Error while write/append {e}")